In [1]:
import sys, os, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

%load_ext autoreload
%autoreload 2
%aimport -src.rl.models.cql_with_logprobs

repo_root = Path("/home/jake/Documents/AI_Clinician/ai_clinician_pediatric_rl")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import src.rl.compare_floor_bound_five_seeds as cmp
import src.rl.diagnose_vaso_saturation as dvs
import src.rl.evaluate_vaso_reward_sweep as vrs

from src.rl.analysis import runs as R

pd.set_option("display.float_format", "{:.4g}".format)
pd.set_option("display.max_columns", 60)

In [2]:
ddh_five_dir = "/home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/picu/with_actions/five_seeds"
rl_base = "/home/jake/Documents/AI_Clinician/models/rl_models/choc"
sarsa_dir = os.path.join(rl_base, "basic_q/split_0/sepsis/intake-vis/picu_patients_health_score_from_picu_dataset/with_actions/health_score_change_only/five_seeds")
cql_base = os.path.join(rl_base, "cql_with_logprobs/split_0/sepsis/intake-vis/picu_patients_health_score_from_picu_dataset/with_actions/health_score_change_only")
cql_dirs = {"no_floor_bound": os.path.join(cql_base, "no_floor_bound"),
            "floor_bound": os.path.join(cql_base, "floor_bound")}

SAVE_DIR = os.path.join(cql_base, "floor_bound_comparison_five_seeds")
os.makedirs(SAVE_DIR, exist_ok=True)

In [3]:
R.set_path_mappings([
    ("/rds/general/user/jormond/home/projects/ai_clinician",
     "/home/jake/Documents/AI_Clinician/models"),
])

study = R.make_study({"no_floor_bound": cql_dirs["no_floor_bound"],
                      "floor_bound": cql_dirs["floor_bound"]}, name="floor comparison")
display(study.table(["cql_alpha", "vaso_mean_floor", "vaso_gate_logit_bound",
                     "fluid_mean_reg_scale", "vaso_mean_reg_scale"]))
_ = study.check(expected_between=["_group", "vaso_mean_floor", "vaso_mean_floor_scale",
                                  "vaso_gate_logit_bound", "vaso_gate_logit_bound_scale"])
study.data_identities()

  overrides applied under /home/jake/Documents/AI_Clinician/models/rl_models/choc/cql_with_logprobs/split_0/sepsis/intake-vis/picu_patients_health_score_from_picu_dataset/with_actions/health_score_change_only/no_floor_bound:
    vaso_reward_coefficient: saved=0.1 -> used=0.0  [rule vaso_reward_gate: add_reward_to_vaso_dosing is False]  seeds=[0, 1, 2, 3, 4]
    fluid_mean_reg_scale: saved=MISSING -> used=0.05  [run_overrides.yaml: hardcoded in get_trainer for these runs (not recorded in results.json)]  seeds=[0, 1, 2, 3, 4]
    vaso_mean_reg_scale: saved=0.0 -> used=0.05  [run_overrides.yaml: hardcoded in get_trainer for these runs (not recorded in results.json)]  seeds=[0, 1, 2, 3, 4]
  overrides applied under /home/jake/Documents/AI_Clinician/models/rl_models/choc/cql_with_logprobs/split_0/sepsis/intake-vis/picu_patients_health_score_from_picu_dataset/with_actions/health_score_change_only/floor_bound:
    vaso_reward_coefficient: saved=0.1 -> used=0.0  [rule vaso_reward_gate: add_rew

family  cql_alpha  vaso_mean_floor  vaso_gate_logit_bound  \
group          seed                                                             
floor_bound    0       cql      0.022           -2.939                      4   
               1       cql      0.022           -2.939                      4   
               2       cql      0.022           -2.939                      4   
               3       cql      0.022           -2.939                      4   
               4       cql      0.022           -2.939                      4   
no_floor_bound 0       cql      0.022              NaN                    NaN   
               1       cql      0.022              NaN                    NaN   
               2       cql      0.022              NaN                    NaN   
               3       cql      0.022              NaN                    NaN   
               4       cql      0.022              NaN                    NaN   

                     fluid_mean_reg_scale  vaso_mean_reg_scale  n_overrides  \
group          seed                                                           
floor_bound    0                     0.05                 0.05            1   
               1                     0.05                 0.05            1   
               2                     0.05                 0.05            1   
               3                     0.05                 0.05            1   
               4                     0.05                 0.05            1   
no_floor_bound 0                     0.05                 0.05            3   
               1                     0.05                 0.05            3   
               2                     0.05                 0.05            3   
               3                     0.05                 0.05            3   
               4                     0.05                 0.05            3   

                                                                  data  \
group          seed                                                      
floor_bound    0     state seed 0, shared reward [choc_dynamic_deep...   
               1     state seed 1, shared reward [choc_dynamic_deep...   
               2     state seed 2, shared reward [choc_dynamic_deep...   
               3     state seed 3, shared reward [choc_dynamic_deep...   
               4     state seed 4, shared reward [choc_dynamic_deep...   
no_floor_bound 0     state seed 0, shared reward [choc_dynamic_deep...   
               1     state seed 1, shared reward [choc_dynamic_deep...   
               2     state seed 2, shared reward [choc_dynamic_deep...   
               3     state seed 3, shared reward [choc_dynamic_deep...   
               4     state seed 4, shared reward [choc_dynamic_deep...   

                                                               run_dir  
group          seed                                                     
floor_bound    0     /home/jake/Documents/AI_Clinician/models/rl_mo...  
               1     /home/jake/Documents/AI_Clinician/models/rl_mo...  
               2     /home/jake/Documents/AI_Clinician/models/rl_mo...  
               3     /home/jake/Documents/AI_Clinician/models/rl_mo...  
               4     /home/jake/Documents/AI_Clinician/models/rl_mo...  
no_floor_bound 0     /home/jake/Documents/AI_Clinician/models/rl_mo...  
               1     /home/jake/Documents/AI_Clinician/models/rl_mo...  
               2     /home/jake/Documents/AI_Clinician/models/rl_mo...  
               3     /home/jake/Documents/AI_Clinician/models/rl_mo...  
               4     /home/jake/Documents/AI_Clinician/models/rl_mo...

Study 'floor comparison': 2 group(s), 10 run(s), seeds [0, 1, 2, 3, 4]
  within floor_bound:
      ratio_of_fluid_to_vaso                   {0: None, 1: 20.144277572631836, 2: 20.144277572631836, 3: 20.144277572631836, 4: 20.144277572631836}
  within no_floor_bound:
      ratio_of_fluid_to_vaso                   {0: None, 1: 20.144277572631836, 2: 20.144277572631836, 3: 20.144277572631836, 4: 20.144277572631836}
  between groups:
      _group                                   {'floor_bound': 'floor_bound', 'no_floor_bound': 'no_floor_bound'}
      vaso_gate_logit_bound                    {'floor_bound': 4.0, 'no_floor_bound': None}
      vaso_gate_logit_bound_scale              {'floor_bound': 0.05, 'no_floor_bound': None}
      vaso_mean_floor                          {'floor_bound': -2.939, 'no_floor_bound': None}
      vaso_mean_floor_scale                    {'floor_bound': 0.05, 'no_floor_bound': None}
  PROBLEMS:
    - within floor_bound: ['ratio_of_fluid_to_vaso']
    - within n

{('/home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/picu/with_actions/five_seeds/choc_dynamic_deephit_transformer_discharge_mortality_tte_standard_split_0_20260925_181749_006821_job4167594.pbs-7',
  0,
  '/home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/picu/with_actions/five_seeds/choc_dynamic_deephit_transformer_discharge_mortality_tte_standard_split_0_20260925_181749_006821_job4167594.pbs-7',
  None,
  'five_seeds'): DataIdentity(state_source='/home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/picu/with_actions/five_seeds/choc_dynamic_deephit_transformer_discharge_mortality_tte_standard_split_0_20260925_181749_006821_job4167594.pbs-7', state_seed=0, reward_source='/home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/picu/with_actions/five_seeds/choc_dynamic

In [5]:
from src.rl.analysis import runs as R, models as M
R.set_path_mappings([("/rds/general/user/jormond/home/projects/ai_clinician",
                      "/home/jake/Documents/AI_Clinician/models")])

study = R.make_study({"no_floor_bound": cql_dirs["no_floor_bound"],
                      "floor_bound": cql_dirs["floor_bound"]}, name="floor comparison")
cql_models = M.load_study_models(study)

sarsa_study = R.make_study(sarsa_dir, name="SARSA")
sarsa_models = M.load_study_models(sarsa_study)

m = cql_models[("floor_bound",)][0]
print(m, m.reward_scale, M.infer_state_dim(m.record))

  overrides applied under /home/jake/Documents/AI_Clinician/models/rl_models/choc/cql_with_logprobs/split_0/sepsis/intake-vis/picu_patients_health_score_from_picu_dataset/with_actions/health_score_change_only/no_floor_bound:
    vaso_reward_coefficient: saved=0.1 -> used=0.0  [rule vaso_reward_gate: add_reward_to_vaso_dosing is False]  seeds=[0, 1, 2, 3, 4]
    fluid_mean_reg_scale: saved=MISSING -> used=0.05  [run_overrides.yaml: hardcoded in get_trainer for these runs (not recorded in results.json)]  seeds=[0, 1, 2, 3, 4]
    vaso_mean_reg_scale: saved=0.0 -> used=0.05  [run_overrides.yaml: hardcoded in get_trainer for these runs (not recorded in results.json)]  seeds=[0, 1, 2, 3, 4]
  overrides applied under /home/jake/Documents/AI_Clinician/models/rl_models/choc/cql_with_logprobs/split_0/sepsis/intake-vis/picu_patients_health_score_from_picu_dataset/with_actions/health_score_change_only/floor_bound:
    vaso_reward_coefficient: saved=0.1 -> used=0.0  [rule vaso_reward_gate: add_rew

In [7]:
from src.rl.analysis import data as D

store = D.DataStore(keep=("val",))
paired = store.pair(cql_models, split="val")             # 5 identities shared by both configs
sarsa_paired = store.pair(sarsa_models, split="val")     # reuses them if the reward settings match
display(store.summary())

val_by_seed = {s: p for s, (m, p) in paired[("floor_bound",)].items()}
display(D.check_rows_aligned(val_by_seed))

display(D.bellman_check({s: m for s, (m, p) in paired[("floor_bound",)].items()}, val_by_seed))
display(D.bellman_check({s: m for s, (m, p) in sarsa_paired[()].items()}, val_by_seed))

  building processors for state seed 0, shared reward [choc_dynamic_deephit_transformer_discharge_mortality_tte_standard_split_0_20260925_181749_006821_job4167594.pbs-7]  (splits: val)


using shifted actions for RL dataset
Using health_score reward type
using shifted actions for RL dataset
Using health_score reward type
using shifted actions for RL dataset
Using health_score reward type
  building processors for state seed 1, shared reward [choc_dynamic_deephit_transformer_discharge_mortality_tte_standard_split_0_20260925_181749_006821_job4167594.pbs-7]  (splits: val)


using shifted actions for RL dataset
Using health_score reward type
using shifted actions for RL dataset
Using health_score reward type
using shifted actions for RL dataset
Using health_score reward type
  building processors for state seed 2, shared reward [choc_dynamic_deephit_transformer_discharge_mortality_tte_standard_split_0_20260925_181749_006821_job4167594.pbs-7]  (splits: val)


using shifted actions for RL dataset
Using health_score reward type
using shifted actions for RL dataset
Using health_score reward type
using shifted actions for RL dataset
Using health_score reward type
  building processors for state seed 3, shared reward [choc_dynamic_deephit_transformer_discharge_mortality_tte_standard_split_0_20260925_181749_006821_job4167594.pbs-7]  (splits: val)


using shifted actions for RL dataset
Using health_score reward type
using shifted actions for RL dataset
Using health_score reward type
using shifted actions for RL dataset
Using health_score reward type
  building processors for state seed 4, shared reward [choc_dynamic_deephit_transformer_discharge_mortality_tte_standard_split_0_20260925_181749_006821_job4167594.pbs-7]  (splits: val)


...

using shifted actions for RL dataset
Using health_score reward type
using shifted actions for RL dataset
Using health_score reward type
using shifted actions for RL dataset
Using health_score reward type


,data,splits_cached,n_val_rows
0,"state seed 0, shared reward [choc_dynamic_deep...",val,90152
1,"state seed 1, shared reward [choc_dynamic_deep...",val,90152
2,"state seed 2, shared reward [choc_dynamic_deep...",val,90152
3,"state seed 3, shared reward [choc_dynamic_deep...",val,90152
4,"state seed 4, shared reward [choc_dynamic_deep...",val,90152
5,"state seed 0, shared reward [choc_dynamic_deep...",val,90152
6,"state seed 1, shared reward [choc_dynamic_deep...",val,90152
7,"state seed 2, shared reward [choc_dynamic_deep...",val,90152
8,"state seed 3, shared reward [choc_dynamic_deep...",val,90152
9,"state seed 4, shared reward [choc_dynamic_deep...",val,90152


,n_rows,same_index_ptr,same_actions,same_health_score,same_rewards,same_states
0,90152,True,True,True,True,True
1,90152,True,True,True,True,False
2,90152,True,True,True,True,False
3,90152,True,True,True,True,False
4,90152,True,True,True,True,False


,data 0,data 1,data 2,data 3,data 4,best_match,matches_own
model 0,0.2131,0.339,0.4123,0.3174,0.4339,0,True
model 1,0.6823,0.4413,0.5125,0.3819,0.5815,3,False
model 2,0.2911,0.2602,0.2046,0.3023,0.3241,2,True
model 3,0.3346,0.5207,0.3684,0.2583,0.4179,3,True
model 4,0.471,0.4135,0.3641,0.3641,0.2437,4,True


,data 0,data 1,data 2,data 3,data 4,best_match,matches_own
model 0,0.5416,0.4414,0.5501,0.4404,0.3903,4,False
model 1,0.4701,0.511,0.4176,0.3074,0.4926,3,False
model 2,0.3447,0.3157,0.5103,0.4028,0.431,1,False
model 3,0.3736,0.5095,0.3363,0.5824,0.4769,2,False
model 4,0.6033,0.4473,0.5116,0.4828,0.4953,1,False


In [8]:
m = cql_models[("floor_bound",)][0]
cm = m.config_manager
print("config attached:", cm is not None)
if cm is not None:
    print("window_length in training_config:", "window_length" in cm.training_config)
    print("training_config keys:", sorted(cm.training_config)[:40])
print("window_length in run hyperparams:", m.record.effective.get("window_length"))
print("window_length in loader hyperparams:", m.loader_extras.get("loader_hyperparams", {}).get("window_length"))

# what the comparison notebook used successfully: the config returned directly by load_actor_and_critic
from src.rl.evaluate_terminal_reward_models import load_actor_and_critic
_, _, cm_direct, hp_direct = load_actor_and_critic(model_dir=m.record.run_dir, model_type="cql", state_dim=128)
print("direct loader config has window_length:", "window_length" in cm_direct.training_config)

config attached: True
window_length in training_config: False
training_config keys: ['action_mode', 'actions_shift', 'actor_hidden_dim', 'actor_loss_gate_entropy_scale', 'actor_loss_mag_entropy_scale', 'actor_loss_q_scale', 'add_reward_to_vaso_dosing', 'batch_size', 'cql_alpha', 'critic_hidden_dim', 'device', 'discount', 'dropout', 'early_stopping_flag', 'eval_freq', 'fluid_mean_reg_scale', 'gate_temp_end', 'gate_temp_start', 'grad_clip', 'group', 'health_score_hourly_change_clip', 'health_score_hourly_change_scale', 'health_score_scale', 'include_next_actions', 'learning_rate_actor', 'learning_rate_critic', 'loss_metric', 'min_delta', 'n_episodes', 'n_step', 'name', 'noise_clip', 'num_samples', 'patience', 'policy_freq', 'policy_noise', 'project', 'ratio_of_fluid_to_vaso', 'reward_type', 'seed']
window_length in run hyperparams: None
window_length in loader hyperparams: None
Loading config manager from: /home/jake/Documents/AI_Clinician/models/rl_models/choc/cql_with_logprobs/split_0/